# Lab 2: Generador de Imágenes con Agentes (Patrón Orquestador)

Un sistema multi-agente con patrón **orquestador**: un agente central controla todo el flujo, los workers nunca hablan entre sí.

**Arquitectura:**
- `orchestrator_agent` — punto central, delega a workers y decide el siguiente paso
- `writer_agent` — worker que escribe texto, devuelve resultado al orchestrator
- `reviewer_agent` — worker que revisa texto, devuelve veredicto al orchestrator
- Los workers tienen `handoffs=[]` — solo el orchestrator decide el flujo

**Flujo:**
```
Usuario → orchestrator → writer (escribe) → orchestrator
                       → reviewer (evalúa) → orchestrator
                       → si rechaza: writer otra vez → orchestrator → reviewer
                       → si aprueba: orchestrator llama generate_image tool
                       → imagen al usuario
```

**Diferencias clave con Lab 1 (prompt chaining):**
- Orquestador: un agente central decide dinámicamente qué worker llamar
- Uso de herramientas: el orchestrator llama a la generación de imagen como tool
- Flujo dinámico: el LLM decide cuándo el texto está listo, no un pipeline fijo

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
    val = os.getenv(key)
    if val:
        os.environ[key] = val
        print(f"{key}: {val[:8]}...{val[-4:]}")
    else:
        print(f"{key} not found.")

os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")

In [ ]:
import tempfile
from google import genai
from google.genai import types
from agents import Agent, Runner, function_tool

# Cliente Gemini para generación de imágenes
gemini_client = genai.Client()
GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image"

IMAGE_PROMPT_TEMPLATE = """
16:9 landscape hand-drawn whiteboard illustration on off-white paper with faint blueprint grid lines.
COMPOSITION: All elements centered with 15% margin on all sides.

Subject: {approved_text}

VISUAL REQUIREMENTS:
- Each component represented by a DISTINCTIVE ICON (gears, brains, clouds, envelopes, shields, etc.) inside or above its labeled box
- Boxes connected by sketchy arrows with small annotations on the arrows describing the data flow
- Small decorative doodles: stars, lightbulbs, checkmarks, dotted trails, tiny sparkles
- Color-coded watercolor fills: each box a different soft color (blue, green, orange, coral, purple)
- Short Spanish labels in handwritten script inside each box

Style: hand-drawn pen-and-ink sketch with loose cross-hatching and watercolor accents.
Analog, warm, textured feel — like a creative brainstorming whiteboard. No digital vectors. No title.
"""

IMAGE_GEN_CONFIG = types.GenerateContentConfig(
    response_modalities=["IMAGE"],
    image_config=types.ImageConfig(aspect_ratio="16:9"),
)

last_image_path = None


@function_tool
def generate_image(approved_text: str) -> str:
    """Genera una imagen estilo whiteboard a partir de texto aprobado.
    Llamar SOLO después de que el texto haya sido revisado y aprobado.

    Args:
        approved_text: La descripción de texto revisada y aprobada para visualizar.

    Returns:
        Ruta al archivo de imagen generado.
    """
    global last_image_path

    image_prompt = IMAGE_PROMPT_TEMPLATE.format(approved_text=approved_text)

    response = gemini_client.models.generate_content(
        model=GEMINI_IMAGE_MODEL,
        contents=image_prompt,
        config=IMAGE_GEN_CONFIG,
    )

    image_bytes = None
    for part in response.candidates[0].content.parts:
        if part.inline_data and part.inline_data.data:
            image_bytes = part.inline_data.data
            break

    if not image_bytes:
        return "ERROR: No se generó ninguna imagen."

    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    tmp.write(image_bytes)
    tmp.close()
    last_image_path = tmp.name
    return f"Imagen generada correctamente en: {tmp.name}"


print("Herramienta 'generate_image' definida.")

In [ ]:
# --- Definición de Agentes (Patrón Orquestador) ---
# Solo el orchestrator tiene handoffs. Los workers devuelven su resultado al orchestrator.

# Worker: escritor de texto (sin handoffs — solo produce output)
writer_agent = Agent(
    name="writer_agent",
    model="gpt-4.1-nano",
    handoff_description="Writes detailed technical descriptions. Returns the text to the orchestrator.",
    instructions="""
You are an expert technical communicator. Given a topic (or feedback on a previous draft),
write a rich, detailed description that captures the key concepts, relationships, and flow.

Include:
- The main components or actors involved
- How they interact or connect
- The sequence or flow of operations
- Visual details that help someone draw a diagram

Write 3-5 detailed sentences. Be specific and descriptive, not abstract.
Return your text directly — the orchestrator will handle the next step.
""",
)

# Worker: revisor de texto (sin handoffs — solo produce output)
reviewer_agent = Agent(
    name="reviewer_agent",
    model="gpt-4.1-mini",
    handoff_description="Reviews text for completeness and visual clarity. Returns verdict to the orchestrator.",
    instructions="""
You are a Senior Technical Reviewer. You receive text descriptions and evaluate them for:
1. Are all key components named and described?
2. Are relationships and data flows explicit?
3. Is the sequence of operations clear?
4. Is there enough visual detail for someone to draw a diagram?

Return your verdict:
- If APPROVED: respond with "APPROVED:" followed by the final text (keep or enrich it).
- If NEEDS REVISION: respond with "NEEDS REVISION:" followed by specific feedback on what to fix.

You may only reject ONCE — if this is the second review, approve it.
""",
)

# Orquestador: controla todo el flujo, delega a workers, llama a la tool
orchestrator_agent = Agent(
    name="orchestrator_agent",
    model="gpt-4.1-mini",
    instructions="""
You are the ORCHESTRATOR of an image generation pipeline. You control the entire workflow.
Workers (writer_agent, reviewer_agent) NEVER talk to each other — only you decide the flow.

WORKFLOW:
1. Receive a TOPIC from the user → hand off to writer_agent to create text.
2. Receive text from writer → hand off to reviewer_agent to evaluate it.
3. Receive verdict from reviewer:
   - If reviewer says "NEEDS REVISION": hand off to writer_agent with the feedback.
     Then send the new text to reviewer_agent again.
   - If reviewer says "APPROVED": call the generate_image tool with the approved text.
4. After image is generated, report the result to the user.

RULES:
- You MUST mediate all communication — workers never hand off to each other.
- You MUST use the generate_image tool after text is approved.
- Maximum 1 revision round — if reviewer rejects twice, use the text as-is.
""",
    tools=[generate_image],
    handoffs=[writer_agent, reviewer_agent],
)

print("Agentes definidos (patrón orquestador):")
print(f"  orchestrator_agent → delega a: writer_agent, reviewer_agent")
print(f"  writer_agent → handoffs: ninguno (devuelve al orchestrator)")
print(f"  reviewer_agent → handoffs: ninguno (devuelve al orchestrator)")
print(f"  orchestrator_agent tiene herramienta: generate_image")

In [ ]:
import traceback
from pathlib import Path
import gradio as gr

RESTYLE_PROMPT = """
Recreate this image as a 16:9 landscape hand-drawn whiteboard illustration on off-white paper with faint blueprint grid lines.

TEXT RULES:
- Translate ONLY descriptive text and labels to Spanish
- DO NOT translate proper names, brand names, product names — keep them EXACTLY as they appear
- DO NOT translate technical acronyms (API, SDK, HTTP, REST, etc.)

LOGO RULES:
- Leave logos UNTOUCHED — same shape, same original colors, no modifications
- Do NOT add any text near or around logos
- Do NOT recolor logos with watercolor or any other fill
- Do NOT invent text to describe or label what a logo is

Keep the same concepts and relationships, but redraw NON-LOGO elements in this style:
- Hand-drawn pen-and-ink sketch with loose cross-hatching
- Subtle watercolor color accents for boxes and arrows only
- Small decorative doodles: stars, lightbulbs, checkmarks
- Handwritten script for labels
- Analog whiteboard feel. No digital vectors, no title header.
"""


# --- Pestaña 1: Generación con agentes ---

async def generate_from_topic(message, history):
    try:
        yield "Iniciando pipeline de agentes..."

        result = await Runner.run(orchestrator_agent, input=message)

        if last_image_path and Path(last_image_path).exists():
            yield gr.Image(last_image_path)
        else:
            yield f"**Resultado del agente:**\n{result.final_output}"

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


# --- Pestaña 2: Regenerar imagen subida ---

def restyle_image(message, history):
    global last_image_path

    try:
        text = message.get("text", "").strip()
        files = message.get("files", [])

        if not files:
            yield "Pega o sube una imagen para regenerar."
            return

        image_path = files[0]
        image_bytes = Path(image_path).read_bytes()

        prompt = RESTYLE_PROMPT
        if text:
            prompt += f"\n\nAdditional instructions: {text}"

        yield "Regenerando imagen en estilo whiteboard 16:9 (Gemini)..."

        response = gemini_client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=[
                types.Content(
                    parts=[
                        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
                        types.Part.from_text(text=prompt),
                    ]
                )
            ],
            config=IMAGE_GEN_CONFIG,
        )

        result_bytes = None
        for part in response.candidates[0].content.parts:
            if part.inline_data and part.inline_data.data:
                result_bytes = part.inline_data.data
                break

        if not result_bytes:
            yield "**Error:** No se generó imagen en la respuesta de Gemini."
            return

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        tmp.write(result_bytes)
        tmp.close()
        last_image_path = tmp.name

        yield gr.Image(tmp.name)

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


def download_last_image():
    return last_image_path


# --- Interfaz ---

with gr.Blocks() as demo:
    gr.Markdown("# Lab 2: Generador de Imágenes con Agentes")
    gr.Markdown("Sistema multi-agente con **handoffs** y **uso de herramientas** | Texto: OpenAI | Imagen: Gemini")

    with gr.Tabs():
        with gr.TabItem("Generar desde tema"):
            gr.ChatInterface(
                fn=generate_from_topic,
                examples=["How multi-agent systems coordinate tasks using handoffs"],
                multimodal=False,
            )

        with gr.TabItem("Regenerar imagen"):
            gr.Markdown("Pega (Ctrl+V) o sube una imagen y se regenerará en estilo whiteboard con textos en español.")
            gr.ChatInterface(
                fn=restyle_image,
                multimodal=True,
            )

    download_btn = gr.DownloadButton("Descargar última imagen", variant="primary")
    download_btn.click(fn=download_last_image, outputs=download_btn)

demo.launch(inline=True)